# GEFS Operational Download Links

<div style="background-color: lightblue; padding: 10px; border: 1px solid blue;">
  <h3>User Instructions</h3>
  <p>To use: change operational mode to false (for now), input day of interest for each download script (and last cell for the mean file) </p>
</div>

### B file Fast script

In [1]:
%%script echo skipping
import xarray as xr
from herbie import Herbie
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import shutil
import os

# --- Configuration ---
date = "2026-03-03"
members = range(31)  # 0 to 30
product = "atmos.5b"
forecast_hours = list(range(60, 205, 6))  # 60, 66, ..., 204
max_retries = 3

# Cache path
#cache_path = f"/home11/ugrad/2024/dh512351/data/gefs/{date.replace('-', '')}"
cache_path = f"/nfs/spare11/dharkin/research/NWS_Lapenta_Project/Herbie_Cache/{date.replace('-', '')}"
# Subset region
lat_bounds = (40, 50)
lon_bounds = (-105, -90)

# Variables to download
variables = {
    "VVEL:700 mb": "w_700",
    "VVEL:500 mb": "w_500",
    "SPFH:850 mb": "q_850",
    "SPFH:700 mb": "q_700",
    "SPFH:500 mb": "q_500",
    "SPFH:2 m above ground": "sh2",    # ✅ 2-meter specific humidity renamed to match training
    #"PVORT:320 K": "pvort_320K",
    "SFCR": "sfcr",
    "HPBL:surface": "pbl_height",
}




# --- Clear Herbie cache before run ---
if os.path.exists(cache_path):
    print(f"🧹 Clearing Herbie cache at {cache_path}")
    shutil.rmtree(cache_path)
else:
    print(f"ℹ️ Cache path does not exist: {cache_path}")

def load_variable_with_retry(H, var_string, rename):
    for attempt in range(max_retries):
        try:
            ds = H.xarray(var_string)
            short_name = list(ds.data_vars.keys())[0]
            ds = ds.rename({short_name: rename})
            return ds
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                print(f"❌ Failed to load '{var_string}' after {max_retries} tries: {e}")
    return None

def download_member(member):
    member_datasets = []
    for fxx in forecast_hours:
        try:
            H = Herbie(date, model="gefs", product=product, member=member, fxx=fxx, cache_dir=cache_path)
            var_datasets = []
            for var_string, rename in variables.items():
                ds = load_variable_with_retry(H, var_string, rename)
                if ds is not None:
                    var_datasets.append(ds)

            if not var_datasets:
                print(f"⚠️ No variables loaded for member {member}, f{fxx}")
                continue

            ds_merged = xr.merge(var_datasets, compat="override")

            if "step" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"step": [fxx]})
            if "number" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"number": [member]})

            # Crop to region
            ds_merged = ds_merged.sel(
                latitude=slice(lat_bounds[1], lat_bounds[0]),
                longitude=slice(lon_bounds[0] % 360, lon_bounds[1] % 360)
            )

            member_datasets.append(ds_merged)

        except Exception as e_member:
            print(f"❌ Failed to process member {member}, f{fxx}: {e_member}")

    if member_datasets:
        return xr.concat(member_datasets, dim="step", coords="minimal", compat="override")
    else:
        return None

# --- Download all members in parallel ---
all_datasets = []
with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_member, m): m for m in members}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading members"):
        ds = future.result()
        if ds is not None:
            all_datasets.append(ds)

# --- Merge and compute median ---
# --- Merge and compute median ---
# --- Merge, compute median, and interpolate to 0.25° ---
if all_datasets:
    ds_all = xr.concat(all_datasets, dim="number")
    ds_median = ds_all.median(dim="number", keep_attrs=True)

    # Interpolate to 0.25° grid
    target_lats = np.arange(lat_bounds[0], lat_bounds[1] + 0.01, 0.25)
    target_lons = np.arange(lon_bounds[0], lon_bounds[1] + 0.01, 0.25)
    target_grid = {
        "latitude": target_lats[::-1],  # descending for interp
        "longitude": target_lons % 360
    }
    ds_interp = ds_median.interp(target_grid, method="linear")

    # Convert to DataFrame and save
    df = ds_interp.to_dataframe().reset_index()

    # Extract year, month, day from the date string
    year = date[:4]
    month = date[5:7]
    day = date[8:10]

    output_dir_base = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"
    output_dir = os.path.join(output_dir_base, year, month, day)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, f"Median_0.5b_downscaled_{date.replace('-', '')}.parquet")
    df.to_parquet(output_path, engine="pyarrow", index=False)
    print(f"✅ Saved 0.25° interpolated ensemble median Parquet to {output_path}")

else:
    print("❌ No data downloaded successfully.")


skipping


### Automated 0.5b Script 

In [2]:
import xarray as xr
from herbie import Herbie
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import shutil
import os
import logging
from datetime import datetime, timedelta
from pathlib import Path

# ============================================================
# USER SETTINGS
# ============================================================

USE_LATEST = True
HARDCODE_DATE = "2026-03-04 00:00"

members = range(31)
product = "atmos.5b"
forecast_hours = list(range(60, 205, 6))
max_retries = 3

BASE_CACHE_DIR = Path("/nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache")

lat_bounds = (40, 50)
lon_bounds = (-105, -90)

# ============================================================
# VARIABLES
# ============================================================

variables = {
    "VVEL:700 mb": "w_700",
    "VVEL:500 mb": "w_500",
    "SPFH:850 mb": "q_850",
    "SPFH:700 mb": "q_700",
    "SPFH:500 mb": "q_500",
    "SPFH:2 m above ground": "sh2",
    "SFCR": "sfcr",
    "HPBL:surface": "pbl_height",
}

# ============================================================
# LOGGING (LOW NOISE)
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ============================================================
# CYCLE SELECTION (12z preferred, fallback 00z)
# ============================================================

def get_candidate_cycles():
    now = datetime.utcnow()
    today_12z = now.replace(hour=12, minute=0, second=0, microsecond=0)
    today_00z = now.replace(hour=0, minute=0, second=0, microsecond=0)

    if now >= today_12z:
        return [today_12z, today_00z]
    else:
        yesterday = now - timedelta(days=1)
        yesterday_12z = yesterday.replace(hour=12, minute=0, second=0, microsecond=0)
        return [today_00z, yesterday_12z]


def select_cycle():
    if not USE_LATEST:
        selected = datetime.strptime(HARDCODE_DATE, "%Y-%m-%d %H:%M")
        logging.info(f"Using hardcoded cycle {selected:%Y-%m-%d %Hz}")
        return selected

    for cycle in get_candidate_cycles():
        try:
            cache_path = BASE_CACHE_DIR / cycle.strftime("%Y%m%d")
            cache_path.mkdir(parents=True, exist_ok=True)

            H = Herbie(
                cycle,
                model="gefs",
                product=product,
                member=0,
                fxx=forecast_hours[0],
                cache_dir=str(cache_path),
            )

            _ = H.xarray("VVEL:700 mb")
            logging.info(f"Using latest available cycle {cycle:%Y-%m-%d %Hz}")
            return cycle

        except Exception:
            logging.warning(f"{cycle:%Y-%m-%d %Hz} unavailable.")

    raise RuntimeError("No 00z or 12z cycles available.")

# ============================================================
# SELECT CYCLE
# ============================================================

selected_cycle = select_cycle()
date = selected_cycle
cache_path = BASE_CACHE_DIR / selected_cycle.strftime("%Y%m%d")

print("\n==============================")
print("FINAL SELECTED CYCLE")
print("==============================")
print(f"Cycle requested: {selected_cycle:%Y-%m-%d %Hz}")
print(f"Cache: {cache_path}")
print("==============================\n")

# ============================================================
# CLEAR CACHE
# ============================================================

if os.path.exists(cache_path):
    logging.info(f"Clearing cache at {cache_path}")
    shutil.rmtree(cache_path)

os.makedirs(cache_path, exist_ok=True)

# ============================================================
# DOWNLOAD LOGIC
# ============================================================

def load_variable_with_retry(H, var_string, rename):
    for attempt in range(max_retries):
        try:
            ds = H.xarray(var_string)
            short_name = list(ds.data_vars.keys())[0]
            return ds.rename({short_name: rename})
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(1)
    return None


def download_member(member):
    logging.info(f"Starting member {member}")
    member_datasets = []
    failure_count = 0

    for fxx in forecast_hours:
        try:
            H = Herbie(
                date,
                model="gefs",
                product=product,
                member=member,
                fxx=fxx,
                cache_dir=str(cache_path),
                verbose=False,
            )

            var_datasets = []
            for var_string, rename in variables.items():
                ds = load_variable_with_retry(H, var_string, rename)
                if ds is not None:
                    var_datasets.append(ds)

            if not var_datasets:
                failure_count += 1
                continue

            ds_merged = xr.merge(var_datasets, compat="override")

            if "step" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"step": [fxx]})
            if "number" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"number": [member]})

            ds_merged = ds_merged.sel(
                latitude=slice(lat_bounds[1], lat_bounds[0]),
                longitude=slice(lon_bounds[0] % 360, lon_bounds[1] % 360),
            )

            member_datasets.append(ds_merged)

        except Exception:
            failure_count += 1

    logging.info(f"Finished member {member} (failures: {failure_count})")

    if member_datasets:
        return xr.concat(member_datasets, dim="step", coords="minimal", compat="override")

    return None

# ============================================================
# PARALLEL DOWNLOAD
# ============================================================

all_datasets = []

logging.info("Beginning parallel member downloads...")

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_member, m): m for m in members}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Members"):
        ds = future.result()
        if ds is not None:
            all_datasets.append(ds)

logging.info("Download phase complete.")

# ============================================================
# MERGE + MEDIAN + INTERP + SAVE
# ============================================================

if all_datasets:
    logging.info("Merging members...")
    ds_all = xr.concat(all_datasets, dim="number")

    logging.info("Computing ensemble median...")
    ds_median = ds_all.median(dim="number", keep_attrs=True)

    logging.info("Interpolating to 0.25° grid...")
    target_lats = np.arange(lat_bounds[0], lat_bounds[1] + 0.01, 0.25)
    target_lons = np.arange(lon_bounds[0], lon_bounds[1] + 0.01, 0.25)

    ds_interp = ds_median.interp(
        latitude=target_lats[::-1],
        longitude=target_lons % 360,
        method="linear"
    )

    logging.info("Converting to DataFrame...")
    df = ds_interp.to_dataframe().reset_index()

    year = f"{date:%Y}"
    month = f"{date:%m}"
    day = f"{date:%d}"

    output_dir_base = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"
    output_dir = os.path.join(output_dir_base, year, month, day)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(
        output_dir,
        f"Median_0.5b_downscaled_{date:%Y%m%d%H}.parquet"
    )

    df.to_parquet(output_path, engine="pyarrow", index=False)

    print(f"\n✅ Saved ensemble median Parquet to:\n{output_path}\n")

else:
    print("❌ No data downloaded successfully.")

/tmp/ipykernel_4190081/2795616985.py:60: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()


✅ Found ┊ model=gefs ┊ product=atmos.5b ┊ 2026-Mar-05 12:00 UTC F60 ┊ GRIB2 @ aws ┊ IDX @ aws


2026-03-05 17:30:10,030 - INFO - Using latest available cycle 2026-03-05 12z
2026-03-05 17:30:10,031 - INFO - Clearing cache at /nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache/20260305
2026-03-05 17:30:10,033 - INFO - Beginning parallel member downloads...
2026-03-05 17:30:10,034 - INFO - Starting member 0
2026-03-05 17:30:10,035 - INFO - Starting member 1
2026-03-05 17:30:10,036 - INFO - Starting member 2
2026-03-05 17:30:10,037 - INFO - Starting member 3
2026-03-05 17:30:10,037 - INFO - Starting member 4
2026-03-05 17:30:10,039 - INFO - Starting member 5
2026-03-05 17:30:10,042 - INFO - Starting member 6
2026-03-05 17:30:10,044 - INFO - Starting member 7
2026-03-05 17:30:10,046 - INFO - Starting member 8
2026-03-05 17:30:10,047 - INFO - Starting member 9



FINAL SELECTED CYCLE
Cycle requested: 2026-03-05 12z
Cache: /nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache/20260305



2026-03-05 17:30:10,048 - INFO - Starting member 10
Members:   0%|          | 0/31 [00:00<?, ?it/s]2026-03-05 17:30:10,049 - INFO - Starting member 11
2026-03-05 17:31:07,500 - INFO - Finished member 0 (failures: 0)
2026-03-05 17:31:07,585 - INFO - Finished member 5 (failures: 0)
Members:   3%|▎         | 1/31 [00:57<28:47, 57.58s/it]2026-03-05 17:31:07,639 - INFO - Starting member 12
2026-03-05 17:31:07,710 - INFO - Starting member 13
2026-03-05 17:31:08,093 - INFO - Finished member 3 (failures: 0)
2026-03-05 17:31:08,205 - INFO - Finished member 2 (failures: 0)
2026-03-05 17:31:08,242 - INFO - Finished member 6 (failures: 0)
2026-03-05 17:31:08,285 - INFO - Starting member 14
Members:  10%|▉         | 3/31 [00:58<07:04, 15.17s/it]2026-03-05 17:31:08,301 - INFO - Finished member 4 (failures: 0)
2026-03-05 17:31:08,415 - INFO - Starting member 15
Members:  13%|█▎        | 4/31 [00:58<04:29,  9.96s/it]2026-03-05 17:31:08,431 - INFO - Starting member 16
2026-03-05 17:31:08,447 - INFO - S


✅ Saved ensemble median Parquet to:
/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.5b_downscaled_2026030512.parquet



### 0.5 Degree A file

### A File Fast Download Script (with Cache Clearing)

In [3]:
%%script echo skipping
import xarray as xr
from herbie import Herbie
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import shutil
import os

# --- Configuration ---
date = "2026-03-03"
members = range(31)  # 0 to 30
product = "atmos.5"
forecast_hours = list(range(60, 205, 6))  # 60, 66, ..., 204
max_retries = 3

# Cache path
#cache_path = f"/home11/ugrad/2024/dh512351/data/gefs/{date.replace('-', '')}"
cache_path = f"/nfs/spare11/dharkin/research/NWS_Lapenta_Project/Herbie_Cache/{date.replace('-', '')}"
# Subset region
lat_bounds = (40, 50)
lon_bounds = (-105, -90)

# Variables to download
variables = {
    "UGRD:700 mb": "u_700",
    "VGRD:700 mb": "v_700",
    "HGT:700 mb": "hgt_700",
    "TMP:700 mb": "t_700",
    #"RH:700 mb": "rh_700",
    "UGRD:850 mb": "u_850",
    "VGRD:850 mb": "v_850",
    "HGT:850 mb": "hgt_850",
    "TMP:850 mb": "t_850",
    #"RH:850 mb": "rh_850",
    "UGRD:500 mb": "u_500",
    "VGRD:500 mb": "v_500",
    "HGT:500 mb": "hgt_500",
    "TMP:500 mb": "t_500",
    #"RH:500 mb": "rh_500",
    "VVEL:850 mb": "vvel_850",
}

# --- Clear Herbie cache before run ---
if os.path.exists(cache_path):
    print(f"🧹 Clearing Herbie cache at {cache_path}")
    shutil.rmtree(cache_path)
else:
    print(f"ℹ️ Cache path does not exist: {cache_path}")

def load_variable_with_retry(H, var_string, rename):
    for attempt in range(max_retries):
        try:
            ds = H.xarray(var_string)
            short_name = list(ds.data_vars.keys())[0]
            ds = ds.rename({short_name: rename})
            return ds
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                print(f"❌ Failed to load '{var_string}' after {max_retries} tries: {e}")
    return None

def download_member(member):
    member_datasets = []
    for fxx in forecast_hours:
        try:
            H = Herbie(date, model="gefs", product=product, member=member, fxx=fxx, cache_dir=cache_path)
            var_datasets = []
            for var_string, rename in variables.items():
                ds = load_variable_with_retry(H, var_string, rename)
                if ds is not None:
                    var_datasets.append(ds)

            if not var_datasets:
                print(f"⚠️ No variables loaded for member {member}, f{fxx}")
                continue

            ds_merged = xr.merge(var_datasets, compat="override")

            if "step" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"step": [fxx]})
            if "number" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"number": [member]})

            # Crop to region
            ds_merged = ds_merged.sel(
                latitude=slice(lat_bounds[1], lat_bounds[0]),
                longitude=slice(lon_bounds[0] % 360, lon_bounds[1] % 360)
            )

            member_datasets.append(ds_merged)

        except Exception as e_member:
            print(f"❌ Failed to process member {member}, f{fxx}: {e_member}")

    if member_datasets:
        return xr.concat(member_datasets, dim="step", coords="minimal", compat="override")
    else:
        return None

# --- Download all members in parallel ---
all_datasets = []
with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_member, m): m for m in members}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading members"):
        ds = future.result()
        if ds is not None:
            all_datasets.append(ds)

# --- Merge and compute median ---
# --- Merge, compute median, and interpolate to 0.25° ---
# --- Merge and compute median ---
# --- Merge, compute median, and interpolate to 0.25° ---
if all_datasets:
    ds_all = xr.concat(all_datasets, dim="number")
    ds_median = ds_all.median(dim="number", keep_attrs=True)

    # Interpolate to 0.25° grid
    target_lats = np.arange(lat_bounds[0], lat_bounds[1] + 0.01, 0.25)
    target_lons = np.arange(lon_bounds[0], lon_bounds[1] + 0.01, 0.25)
    target_grid = {
        "latitude": target_lats[::-1],  # descending for interp
        "longitude": target_lons % 360
    }
    ds_interp = ds_median.interp(target_grid, method="linear")

    # Convert to DataFrame and save
    df = ds_interp.to_dataframe().reset_index()

    # Extract year, month, day from the date string
    year = date[:4]
    month = date[5:7]
    day = date[8:10]

    output_dir_base = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"
    output_dir = os.path.join(output_dir_base, year, month, day)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, f"Median_0.5a_downscaled_{date.replace('-', '')}.parquet")
    df.to_parquet(output_path, engine="pyarrow", index=False)
    print(f"✅ Saved 0.25° interpolated ensemble median Parquet to {output_path}")

else:
    print("❌ No data downloaded successfully.")



skipping


###  Fully Automated 0.5 degree A Script (needs work to include 12z)

In [4]:
import xarray as xr
from herbie import Herbie
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import shutil
import os
import logging
from datetime import datetime, timedelta
from pathlib import Path

# ============================================================
# USER SETTINGS
# ============================================================

USE_LATEST = True
HARDCODE_DATE = "2026-03-04 00:00"

members = range(31)
product = "atmos.5"
forecast_hours = list(range(60, 205, 6))
max_retries = 3

BASE_CACHE_DIR = Path("/nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache")

lat_bounds = (40, 50)
lon_bounds = (-105, -90)

# ============================================================
# VARIABLES
# ============================================================

variables = {
    "UGRD:700 mb": "u_700",
    "VGRD:700 mb": "v_700",
    "HGT:700 mb": "hgt_700",
    "TMP:700 mb": "t_700",
    "UGRD:850 mb": "u_850",
    "VGRD:850 mb": "v_850",
    "HGT:850 mb": "hgt_850",
    "TMP:850 mb": "t_850",
    "UGRD:500 mb": "u_500",
    "VGRD:500 mb": "v_500",
    "HGT:500 mb": "hgt_500",
    "TMP:500 mb": "t_500",
    "VVEL:850 mb": "vvel_850",
}

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(level=logging.INFO)

# ============================================================
# CYCLE SELECTION
# ============================================================

def get_candidate_cycles():
    now = datetime.utcnow()
    today_12z = now.replace(hour=12, minute=0, second=0, microsecond=0)
    today_00z = now.replace(hour=0, minute=0, second=0, microsecond=0)

    if now >= today_12z:
        return [today_12z, today_00z]
    else:
        yesterday = now - timedelta(days=1)
        yesterday_12z = yesterday.replace(hour=12, minute=0, second=0, microsecond=0)
        return [today_00z, yesterday_12z]

def select_cycle():
    if not USE_LATEST:
        return datetime.strptime(HARDCODE_DATE, "%Y-%m-%d %H:%M")

    for cycle in get_candidate_cycles():
        try:
            cache_path = BASE_CACHE_DIR / cycle.strftime("%Y%m%d")
            cache_path.mkdir(parents=True, exist_ok=True)

            H = Herbie(
                cycle,
                model="gefs",
                product=product,
                member=0,
                fxx=forecast_hours[0],
                cache_dir=str(cache_path),
                verbose=False,
            )

            _ = H.xarray("UGRD:700 mb")
            logging.info(f"Using cycle {cycle:%Y-%m-%d %Hz}")
            return cycle

        except Exception:
            logging.warning(f"{cycle:%Y-%m-%d %Hz} unavailable.")

    raise RuntimeError("No 00z or 12z cycles available.")

# ============================================================
# SELECT CYCLE
# ============================================================

selected_cycle = select_cycle()
date = selected_cycle  # KEEP FULL DATETIME
cache_path = BASE_CACHE_DIR / selected_cycle.strftime("%Y%m%d")

print("\n==============================")
print("FINAL SELECTED CYCLE")
print("==============================")
print(f"Cycle: {selected_cycle:%Y-%m-%d %Hz}")
print(f"Cache: {cache_path}")
print("==============================\n")

# ============================================================
# CLEAR CACHE
# ============================================================

if os.path.exists(cache_path):
    shutil.rmtree(cache_path)

os.makedirs(cache_path, exist_ok=True)

# ============================================================
# DOWNLOAD LOGIC
# ============================================================

def load_variable_with_retry(H, var_string, rename):
    for attempt in range(max_retries):
        try:
            ds = H.xarray(var_string)
            short_name = list(ds.data_vars.keys())[0]
            return ds.rename({short_name: rename})
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(1)
    return None

def download_member(member):
    member_datasets = []

    for fxx in forecast_hours:
        try:
            H = Herbie(
                date,
                model="gefs",
                product=product,
                member=member,
                fxx=fxx,
                cache_dir=str(cache_path),
                verbose=False,   # ← suppress ✅ Found lines
            )

            var_datasets = []
            for var_string, rename in variables.items():
                ds = load_variable_with_retry(H, var_string, rename)
                if ds is not None:
                    var_datasets.append(ds)

            if not var_datasets:
                continue

            ds_merged = xr.merge(var_datasets, compat="override")

            if "step" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"step": [fxx]})
            if "number" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"number": [member]})

            ds_merged = ds_merged.sel(
                latitude=slice(lat_bounds[1], lat_bounds[0]),
                longitude=slice(lon_bounds[0] % 360, lon_bounds[1] % 360),
            )

            member_datasets.append(ds_merged)

        except Exception:
            continue

    if member_datasets:
        return xr.concat(member_datasets, dim="step", coords="minimal", compat="override")
    return None

# ============================================================
# PARALLEL DOWNLOAD
# ============================================================

all_datasets = []

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_member, m): m for m in members}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading members"):
        ds = future.result()
        if ds is not None:
            all_datasets.append(ds)

# ============================================================
# MERGE + MEDIAN + INTERP + SAVE
# ============================================================

if all_datasets:
    ds_all = xr.concat(all_datasets, dim="number")
    ds_median = ds_all.median(dim="number", keep_attrs=True)

    target_lats = np.arange(lat_bounds[0], lat_bounds[1] + 0.01, 0.25)
    target_lons = np.arange(lon_bounds[0], lon_bounds[1] + 0.01, 0.25)

    ds_interp = ds_median.interp(
        latitude=target_lats[::-1],
        longitude=target_lons % 360,
        method="linear"
    )

    df = ds_interp.to_dataframe().reset_index()

    year = f"{date:%Y}"
    month = f"{date:%m}"
    day = f"{date:%d}"

    output_dir_base = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"
    output_dir = os.path.join(output_dir_base, year, month, day)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(
        output_dir,
        f"Median_0.5a_downscaled_{date:%Y%m%d%H}.parquet"
    )

    df.to_parquet(output_path, engine="pyarrow", index=False)
    print(f"✅ Saved ensemble median Parquet to {output_path}")

else:
    print("❌ No data downloaded successfully.")

/tmp/ipykernel_4190081/173323865.py:62: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()
2026-03-05 17:38:32,336 - INFO - Using cycle 2026-03-05 12z



FINAL SELECTED CYCLE
Cycle: 2026-03-05 12z
Cache: /nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache/20260305



✅ Saved ensemble median Parquet to /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.5a_downscaled_2026030512.parquet


### 0.25 Degree (only use if you lose the automatic script)

In [5]:
%%script echo skipping
import xarray as xr
from herbie import Herbie
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import shutil
import os

# --- Configuration ---
date = "2026-03-02"
members = range(31)  # 0 to 30
product = "atmos.25"
forecast_hours = list(range(60, 205, 6))  # 60, 66, ..., 204
max_retries = 3

# Cache path
#cache_path = f"/home11/ugrad/2024/dh512351/data/gefs/{date.replace('-', '')}"
cache_path = f"/nfs/spare11/dharkin/research/NWS_Lapenta_Project/Herbie_Cache/{date.replace('-', '')}"
# Subset region
lat_bounds = (40, 50)
lon_bounds = (-105, -90)

# Variables to download
variables = {
    "TMP:2 m": "t2m",
    #"DPT:2 m": "d2m",
    "UGRD:10 m": "u10",
    "VGRD:10 m": "v10",
    "MSLET:mean sea level": "mslp",
    "TSOIL:0-0.1 m below ground": "tsoil",
    "PWAT:entire atmosphere": "pwat",
    "APCP:surface": "apcp",
    "WEASD:surface": "weasd",
    "GUST:surface": "gust",
}


# --- Clear Herbie cache before run ---
if os.path.exists(cache_path):
    print(f"🧹 Clearing Herbie cache at {cache_path}")
    shutil.rmtree(cache_path)
else:
    print(f"ℹ️ Cache path does not exist: {cache_path}")

def load_variable_with_retry(H, var_string, rename):
    for attempt in range(max_retries):
        try:
            ds = H.xarray(var_string)
            short_name = list(ds.data_vars.keys())[0]
            ds = ds.rename({short_name: rename})
            return ds
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(1)
            else:
                print(f"❌ Failed to load '{var_string}' after {max_retries} tries: {e}")
    return None

def download_member(member):
    member_datasets = []
    for fxx in forecast_hours:
        try:
            H = Herbie(date, model="gefs", product=product, member=member, fxx=fxx, cache_dir=cache_path)
            var_datasets = []
            for var_string, rename in variables.items():
                ds = load_variable_with_retry(H, var_string, rename)
                if ds is not None:
                    var_datasets.append(ds)

            if not var_datasets:
                print(f"⚠️ No variables loaded for member {member}, f{fxx}")
                continue

            ds_merged = xr.merge(var_datasets, compat="override")

            if "step" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"step": [fxx]})
            if "number" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"number": [member]})

            # Crop to region
            ds_merged = ds_merged.sel(
                latitude=slice(lat_bounds[1], lat_bounds[0]),
                longitude=slice(lon_bounds[0] % 360, lon_bounds[1] % 360)
            )

            member_datasets.append(ds_merged)

        except Exception as e_member:
            print(f"❌ Failed to process member {member}, f{fxx}: {e_member}")

    if member_datasets:
        return xr.concat(member_datasets, dim="step", coords="minimal", compat="override")
    else:
        return None

# --- Download all members in parallel ---
all_datasets = []
with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_member, m): m for m in members}
    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading members"):
        ds = future.result()
        if ds is not None:
            all_datasets.append(ds)

# --- Merge and compute median ---
if all_datasets:
    ds_all = xr.concat(all_datasets, dim="number")
    ds_median = ds_all.median(dim="number", keep_attrs=True)

    # Interpolate to 0.25° grid
    target_lats = np.arange(lat_bounds[0], lat_bounds[1] + 0.01, 0.25)
    target_lons = np.arange(lon_bounds[0], lon_bounds[1] + 0.01, 0.25)
    target_grid = {
        "latitude": target_lats[::-1],  # descending for interp
        "longitude": target_lons % 360
    }
    ds_interp = ds_median.interp(target_grid, method="linear")

    # Convert to DataFrame and save
    df = ds_interp.to_dataframe().reset_index()

    # Extract year, month, day from the date string
    year = date[:4]
    month = date[5:7]
    day = date[8:10]

    output_dir_base = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"
    output_dir = os.path.join(output_dir_base, year, month, day)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(output_dir, f"Median_0.25_{date.replace('-', '')}.parquet")
    df.to_parquet(output_path, engine="pyarrow", index=False)
    print(f"✅ Saved 0.25° ensemble median Parquet to {output_path}")

else:
    print("❌ No data downloaded successfully.")


skipping


###  Automated 0.25 degree Script (needs fixing to include 12z runs)

In [6]:
import xarray as xr
from herbie import Herbie
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
import numpy as np
import pandas as pd
import time
import shutil
import os
import logging
from datetime import datetime, timedelta
from pathlib import Path

# ============================================================
# USER SETTINGS
# ============================================================

USE_LATEST = True
HARDCODE_DATE = "2026-03-04 00:00"

members = range(31)
product = "atmos.25"
forecast_hours = list(range(60, 205, 6))
max_retries = 3

BASE_CACHE_DIR = Path("/nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache")

lat_bounds = (40, 50)
lon_bounds = (-105, -90)

# ============================================================
# VARIABLES
# ============================================================

variables = {
    "TMP:2 m": "t2m",
    "UGRD:10 m": "u10",
    "VGRD:10 m": "v10",
    "MSLET:mean sea level": "mslp",
    "TSOIL:0-0.1 m below ground": "tsoil",
    "PWAT:entire atmosphere": "pwat",
    "APCP:surface": "apcp",
    "WEASD:surface": "weasd",
    "GUST:surface": "gust",
}

# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(level=logging.INFO)

# ============================================================
# SELECT CYCLE
# ============================================================

if USE_LATEST:
    now = datetime.utcnow()
    today_12z = now.replace(hour=12, minute=0, second=0, microsecond=0)
    today_00z = now.replace(hour=0, minute=0, second=0, microsecond=0)

    selected_cycle = today_12z if now >= today_12z else today_00z
else:
    selected_cycle = datetime.strptime(HARDCODE_DATE, "%Y-%m-%d %H:%M")

date = selected_cycle  # ← keep full datetime

# ============================================================
# CACHE PATH
# ============================================================

cache_path = BASE_CACHE_DIR / selected_cycle.strftime("%Y%m%d")

print("\n==============================")
print("RUN CONFIGURATION")
print("==============================")
print(f"USE_LATEST: {USE_LATEST}")
print(f"Cycle Used: {selected_cycle:%Y-%m-%d %Hz}")
print(f"Cache Directory: {cache_path}")
print("==============================\n")

# ============================================================
# CLEAR CACHE
# ============================================================

if os.path.exists(cache_path):
    shutil.rmtree(cache_path)

os.makedirs(cache_path, exist_ok=True)

# ============================================================
# DOWNLOAD FUNCTIONS
# ============================================================

def load_variable_with_retry(H, var_string, rename):
    for attempt in range(max_retries):
        try:
            ds = H.xarray(var_string)
            short_name = list(ds.data_vars.keys())[0]
            return ds.rename({short_name: rename})
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(1)
    return None


def download_member(member):
    member_datasets = []

    for fxx in forecast_hours:
        try:
            H = Herbie(
                date,
                model="gefs",
                product=product,
                member=member,
                fxx=fxx,
                cache_dir=str(cache_path),
                verbose=False,   # ← suppress ✅ Found lines
            )

            var_datasets = []

            for var_string, rename in variables.items():
                ds = load_variable_with_retry(H, var_string, rename)
                if ds is not None:
                    var_datasets.append(ds)

            if not var_datasets:
                continue

            ds_merged = xr.merge(var_datasets, compat="override")

            if "step" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"step": [fxx]})
            if "number" not in ds_merged.coords:
                ds_merged = ds_merged.expand_dims({"number": [member]})

            ds_merged = ds_merged.sel(
                latitude=slice(lat_bounds[1], lat_bounds[0]),
                longitude=slice(lon_bounds[0] % 360, lon_bounds[1] % 360),
            )

            member_datasets.append(ds_merged)

        except Exception:
            continue

    if member_datasets:
        return xr.concat(member_datasets, dim="step", coords="minimal", compat="override")

    return None

# ============================================================
# PARALLEL DOWNLOAD
# ============================================================

all_datasets = []

with ThreadPoolExecutor(max_workers=12) as executor:
    futures = {executor.submit(download_member, m): m for m in members}

    for future in tqdm(as_completed(futures), total=len(futures), desc="Downloading members"):
        ds = future.result()
        if ds is not None:
            all_datasets.append(ds)

# ============================================================
# MERGE + MEDIAN + INTERPOLATE + SAVE
# ============================================================

if all_datasets:
    ds_all = xr.concat(all_datasets, dim="number")
    ds_median = ds_all.median(dim="number", keep_attrs=True)

    target_lats = np.arange(lat_bounds[0], lat_bounds[1] + 0.01, 0.25)
    target_lons = np.arange(lon_bounds[0], lon_bounds[1] + 0.01, 0.25)

    ds_interp = ds_median.interp(
        latitude=target_lats[::-1],
        longitude=target_lons % 360,
        method="linear",
    )

    df = ds_interp.to_dataframe().reset_index()

    year = f"{date:%Y}"
    month = f"{date:%m}"
    day = f"{date:%d}"

    output_dir_base = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"
    output_dir = os.path.join(output_dir_base, year, month, day)
    os.makedirs(output_dir, exist_ok=True)

    output_path = os.path.join(
        output_dir,
        f"Median_0.25_downscaled_{date:%Y%m%d%H}.parquet"
    )

    df.to_parquet(output_path, engine="pyarrow", index=False)

    print(f"✅ Saved 0.25° ensemble median Parquet to {output_path}")

else:
    print("❌ No data downloaded successfully.")

/tmp/ipykernel_4190081/1393902808.py:58: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow()



RUN CONFIGURATION
USE_LATEST: True
Cycle Used: 2026-03-05 12z
Cache Directory: /nfs/spare11/dharkin/NWS_Lapenta_Project/Herbie_Cache/20260305



✅ Saved 0.25° ensemble median Parquet to /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.25_downscaled_2026030512.parquet


### INterpolation Code (unfinished, needs to be automated)

In [8]:
from pathlib import Path
import re

# Base directory
parent_dir = Path("/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data")

# Target date
year = "2026"
month = "03"
day = "05"

date_dir = parent_dir / year / month / day

if not date_dir.exists():
    print("Date directory does not exist.")
else:
    parquet_files = list(date_dir.glob("*.parquet"))

    if not parquet_files:
        print("No parquet files found for this date.")
    else:
        # Extract cycle hour from filename (last 2 digits before .parquet)
        cycle_pattern = re.compile(r"(\d{10})\.parquet$")

        files_with_cycle = []

        for f in parquet_files:
            match = cycle_pattern.search(f.name)
            if match:
                timestamp = match.group(1)
                cycle_hour = int(timestamp[-2:])
                files_with_cycle.append((cycle_hour, f))

        if not files_with_cycle:
            print("No valid cycle timestamps found.")
        else:
            # Get highest cycle hour (12 over 00)
            latest_cycle = max(files_with_cycle, key=lambda x: x[0])[0]

            latest_files = [
                f for hour, f in files_with_cycle if hour == latest_cycle
            ]

            print(f"📁 Latest cycle for {year}-{month}-{day}: {latest_cycle:02d}z\n")

            for f in sorted(latest_files):
                print(f"📄 {f}")

📁 Latest cycle for 2026-03-05: 12z

📄 /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.25_downscaled_2026030512.parquet
📄 /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.5a_downscaled_2026030512.parquet
📄 /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.5b_downscaled_2026030512.parquet


In [9]:
import os
from pathlib import Path
import pandas as pd
from functools import reduce
import re

# Step 1: Define parent directory
parent_dir = Path("/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data")

# Step 2: Find latest date directory (YYYY/MM/DD)
date_dirs = sorted(
    [p for p in parent_dir.glob("*/*/*") if p.is_dir()],
    reverse=True
)

if not date_dirs:
    raise FileNotFoundError("No date directories found.")

most_recent_subdir = date_dirs[0]

print(f"📁 Latest date subdirectory: {most_recent_subdir}")

# Step 3: Get all Parquet files in that subdirectory
all_parquet_files = list(most_recent_subdir.glob("*.parquet"))

if not all_parquet_files:
    raise FileNotFoundError(f"No Parquet files found in {most_recent_subdir}")

# Step 4: Detect latest cycle (12z over 00z)
cycle_pattern = re.compile(r"(\d{10})\.parquet$")

files_with_cycle = []

for f in all_parquet_files:
    match = cycle_pattern.search(f.name)
    if match:
        timestamp = match.group(1)
        cycle_hour = int(timestamp[-2:])
        files_with_cycle.append((cycle_hour, f))

if not files_with_cycle:
    raise ValueError("No valid timestamped parquet files found.")

latest_cycle = max(files_with_cycle, key=lambda x: x[0])[0]

parquet_files = sorted(
    [f for hour, f in files_with_cycle if hour == latest_cycle]
)

print(f"🕒 Using cycle: {latest_cycle:02d}z")
print(f"📦 Merging {len(parquet_files)} files")

if len(parquet_files) < 2:
    raise ValueError(
        f"Only {len(parquet_files)} Parquet file(s) found for {latest_cycle:02d}z. Need at least 2 to merge."
    )

# Step 5: Load each file as a DataFrame
dfs = [pd.read_parquet(f) for f in parquet_files]

# Step 6: Merge on latitude, longitude, and step
df_merged = reduce(
    lambda left, right: pd.merge(
        left,
        right,
        on=["latitude", "longitude", "step"],
        how="outer"
    ),
    dfs
)

# Step 7a: Save to file within the subdirectory (UNCHANGED NAME)
subdir_output_path = most_recent_subdir / "merged_by_latlon_step.parquet"
df_merged.to_parquet(subdir_output_path, index=False)

# Step 7b: Also save to general Operational_data directory (UNCHANGED NAME)
general_output_path = parent_dir / "merged_by_latlon_step.parquet"
df_merged.to_parquet(general_output_path, index=False)

print(f"✅ Merged {len(parquet_files)} files from {most_recent_subdir.name} into:")
print(f"   ├── {subdir_output_path}")
print(f"   └── {general_output_path}")

📁 Latest date subdirectory: /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05
🕒 Using cycle: 12z
📦 Merging 3 files
✅ Merged 3 files from 05 into:
   ├── /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/merged_by_latlon_step.parquet
   └── /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/merged_by_latlon_step.parquet


# Comparison with Training Data

In [10]:
import pandas as pd
from pathlib import Path

# --- Load df1 ---
df1 = pd.read_parquet("/nfs/spare11/dharkin/NWS_Lapenta_Project/yearly_data_files/NorthernPlains_FULL_2015_2019_neighborhood.parquet")

# --- Find the most recent merged file ---
parent_dir = Path("/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data")
all_merged_files = list(parent_dir.rglob("merged_by_latlon_step.parquet"))

if not all_merged_files:
    raise FileNotFoundError("No merged_by_latlon_step.parquet files found in the directory.")

most_recent_merged = max(all_merged_files, key=lambda f: f.stat().st_mtime)
print(f"📁 Most recent merged file: {most_recent_merged}")

# --- Load df2 from most recent merged file ---
df2 = pd.read_parquet(most_recent_merged)

# --- Column Rename Mapping (customize as needed) ---
rename_map = {
    "hgt_700": "gh_700",
    "hgt_850": "gh_850",
    "hgt_500": "gh_500",
    # Add more mappings if needed...
}
df2.rename(columns=rename_map, inplace=True)

# --- Match dtypes (float64 → float32) ---
cols1 = set(df1.columns)
cols2 = set(df2.columns)
in_both = cols1 & cols2

for col in in_both:
    dtype1 = df1[col].dtype
    dtype2 = df2[col].dtype
    if dtype1 == 'float32' and dtype2 == 'float64':
        df2[col] = df2[col].astype('float32')
        print(f"Converted {col} from float64 to float32")

# --- Report Remaining Mismatches ---
dtype_mismatches = []
for col in in_both:
    if df1[col].dtype != df2[col].dtype:
        dtype_mismatches.append((col, df1[col].dtype, df2[col].dtype))

if dtype_mismatches:
    print("\n⚠️ Columns with dtype mismatches after conversion:")
    for col, d1, d2 in dtype_mismatches:
        print(f" - {col}: file1={d1}, file2={d2}")
else:
    print("\n✅ All shared columns have matching dtypes after conversion.")


📁 Most recent merged file: /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/merged_by_latlon_step.parquet
Converted v_500 from float64 to float32
Converted gust from float64 to float32
Converted w_500 from float64 to float32
Converted u_500 from float64 to float32
Converted apcp from float64 to float32
Converted w_700 from float64 to float32
Converted longitude from float64 to float32
Converted t2m from float64 to float32
Converted sfcr from float64 to float32
Converted tsoil from float64 to float32
Converted gh_500 from float64 to float32
Converted t_700 from float64 to float32
Converted u_850 from float64 to float32
Converted v_700 from float64 to float32
Converted v_850 from float64 to float32
Converted t_850 from float64 to float32
Converted gh_700 from float64 to float32
Converted u_700 from float64 to float32
Converted sh2 from float64 to float32
Converted q_500 from float64 to float32
Converted q_850 from float64 to float32
Converted q_700 from floa

In [11]:
rename_dict = {
    "hgt_500": "gh_500",
    "hgt_700": "gh_700",
    "hgt_850": "gh_850",
    "vvel_500": "w_500",
    "vvel_700": "w_700",
    "vvel_850": "w_850",
    "mslp": "msl",
    "u10": "ugrd_10m",
    "v10": "vgrd_10m",
    "weasd": "sdwe",
    "pbl_height": "pbl_hgt",
}
df2.rename(columns=rename_dict, inplace=True)


In [12]:
df2.drop(columns=["theta", "meanSea", "gribfile_projection", "gribfile_projection_x", "gribfile_projection_y", "heightAboveGround", "heightAboveGround_y", "heightAboveGround_x", "isobaricInhPa_x", 'isobaricInhPa_y', 'surface_y', 'surface_x', 'atmosphereSingleLayer', 'valid_time', 'valid_time_x', 'valid_time_y', 'time_x', 'time_y', 'depthBelowLandLayer', ], inplace=True, errors='ignore')
#df2

In [13]:
df2.columns

Index(['step', 'latitude', 'longitude', 't2m', 'ugrd_10m', 'vgrd_10m', 'msl',
       'tsoil', 'pwat', 'apcp', 'sdwe', 'gust', 'u_700', 'v_700', 'gh_700',
       't_700', 'u_850', 'v_850', 'gh_850', 't_850', 'u_500', 'v_500',
       'gh_500', 't_500', 'w_850', 'w_700', 'w_500', 'q_850', 'q_700', 'q_500',
       'sh2', 'sfcr', 'pbl_hgt', 'time'],
      dtype='object')

In [14]:
# Convert 'step' to timedelta
df2["step_timedelta"] = pd.to_timedelta(df2["step"])

# Compute valid time by adding step to init time
df2["valid_time"] = df2["time"] + df2["step_timedelta"]
# Extract total hours from timedelta
df2["forecast_lead_hours"] = df2["step_timedelta"].dt.total_seconds() // 3600

# Convert to integer type
df2["forecast_lead_hours"] = df2["forecast_lead_hours"].astype(int)
df2.drop(columns=["step_timedelta", "step"], inplace=True)
df2.rename(columns={"time": "init_time"}, inplace=True)


In [15]:
#df2["day"] = df2["valid_time"].dt.date
df2

,latitude,longitude,t2m,ugrd_10m,vgrd_10m,msl,tsoil,pwat,apcp,sdwe,...,w_500,q_850,q_700,q_500,sh2,sfcr,pbl_hgt,init_time,valid_time,forecast_lead_hours
0,40.0,255.0,283.075958,-0.303972,1.461208,101626.382812,283.401398,4.700000,0.0,0.0,...,0.112156,0.002100,0.001650,0.000140,0.002230,0.763,753.703552,2026-03-05 12:00:00,2026-03-08 00:00:00,60
1,40.0,255.0,276.163605,3.912354,1.114011,102033.046875,278.818390,3.300000,0.0,0.0,...,0.086575,0.002310,0.000742,0.000324,0.002320,0.763,122.604309,2026-03-05 12:00:00,2026-03-08 06:00:00,66
2,40.0,255.0,274.728333,3.450171,1.123298,101894.843750,275.972992,3.600000,0.0,0.0,...,0.100097,0.002071,0.000970,0.000244,0.002050,0.763,92.402710,2026-03-05 12:00:00,2026-03-08 12:00:00,72
3,40.0,255.0,286.281677,1.170220,1.023920,101114.929688,280.792175,5.701148,0.0,0.0,...,0.296325,0.002253,0.001500,0.000608,0.002270,0.763,858.362488,2026-03-05 12:00:00,2026-03-08 18:00:00,78
4,40.0,255.0,288.634308,4.522034,-0.605916,100444.828125,288.204407,6.600000,0.0,0.0,...,0.193260,0.003030,0.002100,0.000539,0.003012,0.763,827.053345,2026-03-05 12:00:00,2026-03-09 00:00:00,84
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62520,50.0,270.0,268.664825,-1.535134,1.702551,101730.171875,268.190002,5.200000,0.1,23.0,...,0.034462,0.001693,0.000671,0.000218,0.002230,1.089,309.970367,2026-03-05 12:00:00,2026-03-13 00:00:00,180
62521,50.0,270.0,267.092194,-1.550239,1.936638,101497.546875,267.683624,5.700000,0.3,23.0,...,-0.051420,0.001910,0.000950,0.000239,0.002290,1.089,388.399841,2026-03-05 12:00:00,2026-03-13 06:00:00,186
62522,50.0,270.0,266.150635,-1.969260,-0.371174,100963.007812,267.465027,6.200000,0.2,24.0,...,0.050562,0.001820,0.000907,0.000288,0.002140,1.089,316.834381,2026-03-05 12:00:00,2026-03-13 12:00:00,192
62523,50.0,270.0,267.652954,-1.186790,-1.158269,101096.148438,267.959015,4.900000,0.3,24.0,...,0.038493,0.001650,0.000770,0.000185,0.002340,1.089,680.768494,2026-03-05 12:00:00,2026-03-13 18:00:00,198


In [16]:
# For all float64 columns that appear as float32 in file1, downcast to float32 in file2
float32_cols = ['msl', 'w_850', 'ugrd_10m', 'vgrd_10m', 'pbl_hgt', 'sdwe']

for col in float32_cols:
    if col in df2.columns:
        df2[col] = df2[col].astype('float32')

# For datetime, convert to consistent precision (usually datetime64[ns] is fine)
df2['init_time'] = pd.to_datetime(df2['init_time'])
df2['init_time'] = df2['init_time'].astype('datetime64[us]')
df2['valid_time'] = pd.to_datetime(df2['valid_time'])
df2['valid_time'] = df2['valid_time'].astype('datetime64[us]')

In [17]:
# Compare column names
cols1 = set(df1.columns)
cols2 = set(df2.columns)

only_in_1 = cols1 - cols2
only_in_2 = cols2 - cols1
in_both = cols1 & cols2

print("✅ Columns only in file1:", only_in_1)
print("✅ Columns only in file2:", only_in_2)

# Check for dtype mismatches
dtype_mismatches = []
for col in in_both:
    if df1[col].dtype != df2[col].dtype:
        dtype_mismatches.append((col, df1[col].dtype, df2[col].dtype))

if dtype_mismatches:
    print("\n⚠️ Columns with mismatched dtypes:")
    for col, dtype1, dtype2 in dtype_mismatches:
        print(f"  - {col}: file1={dtype1}, file2={dtype2}")
else:
    print("\n✅ All shared columns have matching dtypes.")

✅ Columns only in file1: {'neighborhood_label_75km', 'snowfall_ge_3in', 'snowfall_ge_6in', 'wetbulb_850', 'wetbulb_sfc', 'day', 'snowfall_ge_1in', 'pbl_hgt_missing', 'freezing_level_m', 'snowfall_ge_4in', 'snowfall_nohrsc_m', 'neighborhood_label_75km_3in', 'tsoil_missing', 'sfcr_missing', 'sdwe_missing'}
✅ Columns only in file2: set()

✅ All shared columns have matching dtypes.


In [18]:
import os
import pandas as pd

# Ensure 'init_time' is datetime dtype
df2['init_time'] = pd.to_datetime(df2['init_time'])

# Extract date string (YYYYMMDD)
date_str = df2['init_time'].dt.strftime('%Y%m%d').iloc[0]

# Extract cycle hour (00 or 12)
cycle_hour = df2['init_time'].dt.strftime('%H').iloc[0]
cycle_str = f"{cycle_hour}z"

print(f"🕒 Detected cycle: {cycle_str}")

# Base output directory
base_output_dir = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"

# Build dated output directory
year, month, day = date_str[:4], date_str[4:6], date_str[6:8]
output_dir = os.path.join(base_output_dir, year, month, day)
os.makedirs(output_dir, exist_ok=True)

# Clean out existing files in output directory
for filename in os.listdir(output_dir):
    file_path = os.path.join(output_dir, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)
        print(f"🗑️ Deleted old file: {file_path}")

# Final output file path (SAME naming structure, but cycle-aware)
output_path = os.path.join(
    output_dir,
    f"Operational_GEFS_{cycle_str}_{date_str}.parquet"
)

# Save DataFrame to Parquet with snappy compression
df2.to_parquet(output_path, index=False, compression="snappy")
print(f"✅ Saved DataFrame to {output_path}")

# Optional: delete the merged input file too
merged_file = os.path.join(base_output_dir, "merged_all_by_latlon_step.parquet")
if os.path.exists(merged_file):
    os.remove(merged_file)
    print(f"🗑️ Deleted temporary file: {merged_file}")

🕒 Detected cycle: 12z
🗑️ Deleted old file: /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.5b_downscaled_2026030512.parquet
🗑️ Deleted old file: /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.5a_downscaled_2026030512.parquet
🗑️ Deleted old file: /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Median_0.25_downscaled_2026030512.parquet
🗑️ Deleted old file: /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/merged_by_latlon_step.parquet
✅ Saved DataFrame to /nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Operational_GEFS_12z_20260305.parquet


#### Phase Metrics (for Merging)

In [19]:
import os
import glob
import pandas as pd
import numpy as np
import re

# --- Your existing functions ---
EPS = 0.622  # Rd / Rv

def rh_from_q(T_K, p_Pa, q):
    T_C = T_K - 273.15
    es = 6.112 * np.exp((17.67 * T_C) / (T_C + 243.5))
    e = (q * p_Pa) / (EPS + (1 - EPS) * q) / 100.0
    RH = 100.0 * e / es
    return np.clip(RH, 0.0, 100.0).astype(np.float32)

def wetbulb_stull(T_C, RH):
    Tw = (T_C * np.arctan(0.151977 * np.sqrt(RH + 8.313659))
          + np.arctan(T_C + RH)
          - np.arctan(RH - 1.676331)
          + 0.00391838 * RH**1.5 * np.arctan(0.023101 * RH)
          - 4.686035)
    return Tw.astype(np.float32)

def calculate_phase_metrics(df):
    RH_sfc = rh_from_q(df["t2m"].values, df["msl"].values, df["sh2"].values)
    df["wetbulb_sfc"] = wetbulb_stull(df["t2m"].values - 273.15, RH_sfc)

    p850 = 85000.0
    RH_850 = rh_from_q(df["t_850"].values, p850, df["q_850"].values)
    df["wetbulb_850"] = wetbulb_stull(df["t_850"].values - 273.15, RH_850)

    T_sfc = df["t2m"].values - 273.15
    T_850 = df["t_850"].values - 273.15
    T_700 = df["t_700"].values - 273.15
    z_850 = df["gh_850"].values
    z_700 = df["gh_700"].values

    fl = np.full_like(T_sfc, np.nan, dtype=np.float32)

    mask1 = np.sign(T_sfc) * np.sign(T_850) <= 0
    frac1 = np.divide(0 - T_sfc, T_850 - T_sfc, where=mask1 & (T_850 != T_sfc))
    fl[mask1] = (frac1[mask1] * z_850[mask1]).astype(np.float32)

    mask2 = ~mask1 & (np.sign(T_850) * np.sign(T_700) <= 0)
    frac2 = np.divide(0 - T_850, T_700 - T_850, where=mask2 & (T_700 != T_850))
    fl[mask2] = (
        z_850[mask2] + frac2[mask2] * (z_700[mask2] - z_850[mask2])
    ).astype(np.float32)

    df["freezing_level_m"] = fl

    return df


# ============================================================
# Locate most recent operational parquet file (cycle aware)
# ============================================================

base_dir = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"

# Find ALL Operational_GEFS_*z_*.parquet files
pattern = os.path.join(base_dir, "**", "Operational_GEFS_*z_*.parquet")
all_files = glob.glob(pattern, recursive=True)

if not all_files:
    raise FileNotFoundError("No Operational_GEFS_*z_*.parquet files found.")

# Extract cycle hour from filename and choose highest (12z > 00z)
cycle_pattern = re.compile(r"Operational_GEFS_(\d{2})z_(\d{8})\.parquet")

files_with_cycle = []

for f in all_files:
    match = cycle_pattern.search(os.path.basename(f))
    if match:
        cycle_hour = int(match.group(1))
        date_str = match.group(2)
        files_with_cycle.append((date_str, cycle_hour, f))

if not files_with_cycle:
    raise ValueError("No properly formatted Operational_GEFS files found.")

# Sort by date DESC, then cycle DESC
files_with_cycle.sort(key=lambda x: (x[0], x[1]), reverse=True)

input_parquet = files_with_cycle[0][2]
selected_cycle = files_with_cycle[0][1]

print(f"Loading most recent parquet file:\n{input_parquet}")
print(f"Detected cycle: {selected_cycle:02d}z")

df = pd.read_parquet(input_parquet)

print("Calculating phase metrics...")
df = calculate_phase_metrics(df)

# Extract directory to save output
output_dir = os.path.dirname(input_parquet)

# Extract date from init_time
date_str = pd.to_datetime(df["init_time"].iloc[0]).strftime("%Y%m%d")

output_parquet = os.path.join(
    output_dir,
    f"Operational_GEFS_{selected_cycle:02d}z_{date_str}.parquet"
)

print(f"Saving augmented DataFrame to:\n{output_parquet}")
df.to_parquet(output_parquet, engine="pyarrow", compression="zstd")

print("Done!")

Loading most recent parquet file:
/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Operational_GEFS_12z_20260305.parquet
Detected cycle: 12z
Calculating phase metrics...
Saving augmented DataFrame to:
/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Operational_GEFS_12z_20260305.parquet
Done!


In [20]:
df.columns

Index(['latitude', 'longitude', 't2m', 'ugrd_10m', 'vgrd_10m', 'msl', 'tsoil',
       'pwat', 'apcp', 'sdwe', 'gust', 'u_700', 'v_700', 'gh_700', 't_700',
       'u_850', 'v_850', 'gh_850', 't_850', 'u_500', 'v_500', 'gh_500',
       't_500', 'w_850', 'w_700', 'w_500', 'q_850', 'q_700', 'q_500', 'sh2',
       'sfcr', 'pbl_hgt', 'init_time', 'valid_time', 'forecast_lead_hours',
       'wetbulb_sfc', 'wetbulb_850', 'freezing_level_m'],
      dtype='object')

### Converting to mean file 

In [21]:
import os
import glob
import re
import pandas as pd

# ============================================================
# Locate most recent Operational_GEFS_*z_*.parquet
# ============================================================

base_dir = "/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data"

pattern = os.path.join(base_dir, "**", "Operational_GEFS_*z_*.parquet")
all_files = glob.glob(pattern, recursive=True)

if not all_files:
    raise FileNotFoundError("No Operational_GEFS_*z_*.parquet files found.")

cycle_pattern = re.compile(r"Operational_GEFS_(\d{2})z_(\d{8})\.parquet")

files_with_cycle = []

for f in all_files:
    match = cycle_pattern.search(os.path.basename(f))
    if match:
        cycle_hour = int(match.group(1))
        date_str = match.group(2)
        files_with_cycle.append((date_str, cycle_hour, f))

if not files_with_cycle:
    raise ValueError("No properly formatted Operational_GEFS files found.")

# Sort by date DESC, then cycle DESC
files_with_cycle.sort(key=lambda x: (x[0], x[1]), reverse=True)

input_parquet = files_with_cycle[0][2]
selected_cycle = files_with_cycle[0][1]
date_str = files_with_cycle[0][0]

print(f"Loading most recent file:\n{input_parquet}")
print(f"Detected cycle: {selected_cycle:02d}z")

# ============================================================
# Load operational data
# ============================================================

df_op = pd.read_parquet(input_parquet)

# ============================================================
# Define forecast hour windows for Days 3–8
# ============================================================

day_windows = {
    3: (60, 83),
    4: (84, 107),
    5: (108, 131),
    6: (132, 155),
    7: (156, 179),
    8: (180, 203)
}

exclude_cols = ['init_time', 'latitude', 'longitude', 'day', 'forecast_lead_hours']
feature_cols = [col for col in df_op.columns if col not in exclude_cols]

df_op['init_time'] = pd.to_datetime(df_op['init_time'])

aggregated_dfs = []

for day, (start_hr, end_hr) in day_windows.items():
    df_day = df_op[
        (df_op['forecast_lead_hours'] >= start_hr) &
        (df_op['forecast_lead_hours'] <= end_hr)
    ].copy()

    agg_features = (
        df_day
        .groupby(['init_time', 'latitude', 'longitude'])[feature_cols]
        .mean()
        .reset_index()
    )

    agg_features['day'] = day
    aggregated_dfs.append(agg_features)

final_op_df = pd.concat(aggregated_dfs, ignore_index=True)

print(final_op_df)

# ============================================================
# Save with SAME naming structure (cycle aware)
# ============================================================

output_dir = os.path.dirname(input_parquet)

output_path = os.path.join(
    output_dir,
    f"Operational_GEFS_{selected_cycle:02d}z_{date_str}_mean.parquet"
)

final_op_df.to_parquet(output_path, index=False)

print(f"DataFrame saved to:\n{output_path}")

Loading most recent file:
/nfs/spare11/dharkin/NWS_Lapenta_Project/correct_parquet_data/Operational_data/2026/03/05/Operational_GEFS_12z_20260305.parquet
Detected cycle: 12z
                init_time  latitude  longitude         t2m  ugrd_10m  \
0     2026-03-05 12:00:00      40.0     255.00  280.062378  2.057193   
1     2026-03-05 12:00:00      40.0     255.25  279.474213  1.492287   
2     2026-03-05 12:00:00      40.0     255.50  278.873901  2.106175   
3     2026-03-05 12:00:00      40.0     255.75  278.984009  2.535270   
4     2026-03-05 12:00:00      40.0     256.00  279.076721  2.406387   
...                   ...       ...        ...         ...       ...   
15001 2026-03-05 12:00:00      50.0     269.00  268.270569 -1.123763   
15002 2026-03-05 12:00:00      50.0     269.25  268.039734 -1.226124   
15003 2026-03-05 12:00:00      50.0     269.50  267.846100 -1.302153   
15004 2026-03-05 12:00:00      50.0     269.75  267.467041 -1.603367   
15005 2026-03-05 12:00:00      50.